# Get to Know a Dataset: HeritageGraph

This notebook is a guided tour of the [HeritageGraph](https://registry.opendata.aws/cair-nepal-heritage-graph) dataset, an open knowledge graph dataset for cultural heritage. Its first use case is **Nepal**: release v1.0.0 is an RDF knowledge graph of **1,652 monuments of Nepal** (temples, *patis* and *sattals* (rest houses), Buddhist monastic buildings, stone water spouts (*dhunge dhara*), *chaityas* and more), with their locations, architecture, condition, recorded history and the objects they house. It is built from the monument records of the [Digital Archive of Nepalese Arts and Monuments (DANAM)](https://danam.cats.uni-heidelberg.de), Heidelberg University, and modelled with the CIDOC-CRM-aligned [HeritageGraph ontology](https://cairnepal.github.io/heritagegraphontology/) by [CAIR-Nepal](https://cair-nepal.org).

More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/). Documentation for this dataset is at [github.com/CAIRNepal/heritagegraph-aws-open-data](https://github.com/CAIRNepal/heritagegraph-aws-open-data), and the code that builds it is at [github.com/CAIRNepal/HeritageGraphDataset](https://github.com/CAIRNepal/HeritageGraphDataset).

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The bucket `cair-nepal-heritage-graph` (Region `us-east-1`) has **one top-level prefix per release**, named after the release version (`v1.0.0/`, and later `v1.1.0/`, ...). A release prefix is never changed after it is published, so you can pin your work to a version. Each release prefix holds the same graph in three serializations plus two small metadata files:

| Object | What it is |
|---|---|
| `heritagegraph-nepal.ttl` | the graph as RDF Turtle (human-readable) |
| `heritagegraph-nepal.jsonld` | the same graph as JSON-LD |
| `heritagegraph-nepal.nt.gz` | the same graph as gzipped N-Triples, for bulk loading into a triplestore |
| `metadata.json` | a Schema.org / DCAT description of the release: counts, namespaces, file sizes and checksums |
| `checksums.sha256` | SHA-256 of each file, in `sha256sum` format |

All three serializations contain the same triples, so download only the one that suits your tools. Full documentation, including a data dictionary, is at [github.com/CAIRNepal/heritagegraph-aws-open-data](https://github.com/CAIRNepal/heritagegraph-aws-open-data).

In [ ]:
# This notebook requires the following additional libraries
# (install with your preferred method, e.g. pip or conda):
#
# boto3 >= 1.34.0
# rdflib >= 7.0.0
# pandas >= 2.0.0
# matplotlib >= 3.8.0

# Built-ins
import hashlib
import json
from collections import Counter

# Installed libraries
import boto3
import matplotlib.pyplot as plt
import pandas as pd
from botocore import UNSIGNED
from botocore.config import Config
from rdflib import Graph

Next we define the location of the dataset, create an S3 client, and list the top-level prefixes. The bucket is public, so requests are sent unsigned and no AWS account is needed.

In [ ]:
BUCKET = "cair-nepal-heritage-graph"
RELEASE = "v1.0.0/"   # pin a release; new releases appear as new prefixes

# Public bucket: unsigned requests, no AWS credentials needed
s3 = boto3.client("s3", region_name="us-east-1", config=Config(signature_version=UNSIGNED))

for item in s3.list_objects_v2(Bucket=BUCKET, Delimiter="/")["CommonPrefixes"]:
    print(item["Prefix"])

Inside a release prefix are the five objects described above:

In [ ]:
for obj in s3.list_objects_v2(Bucket=BUCKET, Prefix=RELEASE)["Contents"]:
    print(f"{obj['Size'] / 1e6:8.2f} MB  {obj['Key']}")

`metadata.json` describes the release. It is small, so it is a good first thing to read:

In [ ]:
with s3.get_object(Bucket=BUCKET, Key=RELEASE + "metadata.json")["Body"] as body:
    metadata = json.load(body)

print(metadata["name"], metadata["version"], "published", metadata["datePublished"])
print(json.dumps(metadata["statistics"], indent=2))

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

The data is a **knowledge graph** in the [Resource Description Framework (RDF)](https://www.w3.org/TR/rdf11-primer/). RDF stores facts as *subject – predicate – object* triples, where subjects and predicates are IRIs (web identifiers). For example, a temple, its type, its name and its location look like this in Turtle:

```turtle
res:monument/a2a8efa0-...  a  hg:Temple ;
    rdfs:label        "Illātvāḥ Gaṇeśa Mandira" ;
    hg:danamCode      "SKH0310" ;
    hg:hasMainDeity   res:deity/ganesa ;
    geo:lat 27.7301 ;  geo:long 85.4623 ;
    prov:wasDerivedFrom <https://danam.cats.uni-heidelberg.de/report/a2a8efa0-...> .
```

The same graph is published in three standard RDF serializations:

- **Turtle** (`.ttl`, [spec](https://www.w3.org/TR/turtle/)): compact and readable; the best choice for Python and for reading the data by eye.
- **JSON-LD** (`.jsonld`, [spec](https://www.w3.org/TR/json-ld11/)): RDF as JSON, for web applications and JSON tooling.
- **N-Triples, gzipped** (`.nt.gz`, [spec](https://www.w3.org/TR/n-triples/)): one triple per line; easy to stream, split and bulk load.

We chose RDF because heritage records are highly connected (a monument has events, objects, deities, places and sources, each with its own relationships) and because RDF lets us use shared standards: [CIDOC-CRM](https://cidoc-crm.org/) (ISO 21127, the reference model for cultural heritage), [PROV-O](https://www.w3.org/TR/prov-o/) for provenance, [GeoSPARQL](https://www.ogc.org/standard/geosparql/) for coordinates and the [Getty Art & Architecture Thesaurus](https://www.getty.edu/research/tools/vocabularies/aat/) for architectural styles. Every monument links back to the DANAM page it came from.

**Tooling advice.**
- In Python, [`rdflib`](https://rdflib.readthedocs.io/) loads the whole graph (about 220,000 triples) into memory in a few seconds and runs [SPARQL](https://www.w3.org/TR/sparql11-query/) queries over it. That is what this notebook uses.
- For a SPARQL endpoint shared by many users or applications, load the `.nt.gz` file into a triplestore such as [Amazon Neptune](https://aws.amazon.com/neptune/), [Apache Jena Fuseki](https://jena.apache.org/documentation/fuseki2/) or [GraphDB](https://graphdb.ontotext.com/). The Neptune bulk loader reads gzipped N-Triples from an S3 bucket in the same Region as the cluster, so copy the file into your own bucket in your cluster's Region first.
- `metadata.json` and `checksums.sha256` are plain JSON and text.

**Things to know about this graph.**
- Most entities have both a CIDOC-CRM class and a HeritageGraph class (for example `crm:E22_Human-Made_Object` and `hg:Temple`), so you can query at either level. All monuments are `hg:ArchitecturalStructure`.
- Several properties come in pairs that point in opposite directions (`hg:hasEvent` / `hg:isEventOf`, `crm:P46_is_composed_of` / `crm:P46i_forms_part_of`). Use whichever direction makes your query simpler.
- Some fields hold **several values** where DANAM lists several. For example, 213 monuments have two condition values (such as *Restored* and *Good*), and religious traditions can be combined strings such as `"Hindu, Buddhist"`.
- Place names are kept as DANAM gives them and are **not normalised**: the same municipality can appear as "Lalitpur" and "Lalitpur Metropolitan City".

Before analysing a download, you can check it against `checksums.sha256`:

In [ ]:
# Download the Turtle file into memory (about 19 MB) and verify its checksum
TTL_KEY = RELEASE + "heritagegraph-nepal.ttl"
with s3.get_object(Bucket=BUCKET, Key=TTL_KEY)["Body"] as body:
    ttl_bytes = body.read()

with s3.get_object(Bucket=BUCKET, Key=RELEASE + "checksums.sha256")["Body"] as body:
    expected = dict(line.split()[::-1] for line in body.read().decode().splitlines() if line.strip())

actual = hashlib.sha256(ttl_bytes).hexdigest()
print("checksum OK" if actual == expected["heritagegraph-nepal.ttl"] else "CHECKSUM MISMATCH")

### Q: Can you show us an example of downloading and loading data from your dataset?

We parse the Turtle file we just downloaded into an in-memory `rdflib` graph. (To use the command line instead: `aws s3 cp s3://cair-nepal-heritage-graph/v1.0.0/heritagegraph-nepal.ttl . --no-sign-request`.)

In [ ]:
g = Graph()
g.parse(data=ttl_bytes, format="turtle")
print(f"{len(g):,} triples loaded")

We will query the graph with SPARQL. Here are the prefixes used throughout, and a small helper that runs a query and returns a pandas DataFrame:

In [ ]:
PREFIXES = """
PREFIX hg:        <https://cair-nepal.org/heritagegraph/>
PREFIX crm:       <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX rdfs:      <http://www.w3.org/2000/01/rdf-schema#>
PREFIX geo:       <http://www.w3.org/2003/01/geo/wgs84_pos#>
PREFIX geosparql: <http://www.opengis.net/ont/geosparql#>
PREFIX prov:      <http://www.w3.org/ns/prov#>
PREFIX dcterms:   <http://purl.org/dc/terms/>
PREFIX schema:    <https://schema.org/>
"""

def sparql(query):
    """Run a SPARQL query on g and return the results as a DataFrame of Python values."""
    result = g.query(PREFIXES + query)
    rows = [[None if v is None else v.toPython() for v in row] for row in result]
    return pd.DataFrame(rows, columns=[str(v) for v in result.vars])

How many entities of each HeritageGraph class does the graph hold? Classes are hierarchical, so an entity is counted under each class it has (every `hg:Temple` is also an `hg:ReligiousStructure` and an `hg:ArchitecturalStructure`).

In [ ]:
classes = sparql("""
SELECT ?class (COUNT(?s) AS ?n) WHERE {
  ?s a ?c .
  FILTER(STRSTARTS(STR(?c), STR(hg:)))
  BIND(STRAFTER(STR(?c), STR(hg:)) AS ?class)
} GROUP BY ?class ORDER BY DESC(?n)
""")
classes

Now let's look at a single monument: every property it has, and its value. This shows the structure of a monument record. Values that are IRIs point to other nodes in the graph (its events, objects, assessment, place and deity) or to external resources (the DANAM page and Getty AAT concepts).

In [ ]:
# Pick a monument by its DANAM code
monument = sparql('SELECT ?m WHERE { ?m hg:danamCode "SKH0310" }')["m"][0]

record = sparql(f"""
SELECT ?property ?value WHERE {{ <{monument}> ?p ?value .
  BIND(REPLACE(STR(?p), "^.*[/#]", "") AS ?property) }}
ORDER BY ?property
""")
pd.set_option("display.max_colwidth", 90)
record

Each monument links to its **events** (things that happened to or at it) and the **objects** it contains. Here are the events recorded for the same monument, with their HeritageGraph type. (This reconstruction is typed `TransferOfCustody`: event types in v1.0.0 come from keyword rules and are not always right. The label and description are DANAM's own.)

In [ ]:
sparql(f"""
SELECT ?type ?label ?description WHERE {{
  <{monument}> hg:hasEvent ?e .
  ?e a ?c ; rdfs:label ?label .
  OPTIONAL {{ ?e dcterms:description ?description }}
  FILTER(STRSTARTS(STR(?c), STR(hg:)))
  BIND(STRAFTER(STR(?c), STR(hg:)) AS ?type)
}}
""")

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

First, what kinds of monuments are documented? We count the most specific types (leaving out grouping classes such as `hg:RestHouse`, which covers both *patis* and *sattals*). A monument can have more than one type; monuments with none of these types are counted as "Other".

In [ ]:
SPECIFIC = "hg:Temple, hg:Pati, hg:Sattal, hg:BuddhistMonument, hg:DhungeDhara, hg:Chaitya"
types = sparql(f"""
SELECT ?type (COUNT(DISTINCT ?m) AS ?n) WHERE {{
  ?m a hg:ArchitecturalStructure , ?c .
  FILTER(?c IN ({SPECIFIC}))
  BIND(STRAFTER(STR(?c), STR(hg:)) AS ?type)
}} GROUP BY ?type
""")
other = sparql(f"""
SELECT (COUNT(DISTINCT ?m) AS ?n) WHERE {{
  ?m a hg:ArchitecturalStructure .
  FILTER NOT EXISTS {{ ?m a ?c . FILTER(?c IN ({SPECIFIC})) }}
}}""")["n"][0]
types = pd.concat([pd.DataFrame({"type": ["Other"], "n": [other]}),
                   types.sort_values("n")], ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 5), dpi=100)
ax.barh(types["type"], types["n"], color=["#999999"] + ["#2c7fb8"] * (len(types) - 1))
for y, n in enumerate(types["n"]):
    ax.text(n + 5, y, f"{n:,}", va="center", fontsize=9)
ax.set_title("Nepal monuments in HeritageGraph, by type", fontsize=14, fontweight="bold", loc="left")
ax.set_xlabel("Number of monuments")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Next, *where* they are, and in what condition. 1,626 of the 1,652 monuments have coordinates. We map each one, coloured by the condition DANAM recorded. Where a monument has two condition values, we use the worse one.

DANAM documents mainly the Kathmandu Valley, so we draw the whole country first and then zoom in on the valley.

In [ ]:
points = sparql("""
SELECT ?m ?lat ?long (GROUP_CONCAT(STR(?c); separator="|") AS ?conditions) WHERE {
  ?m a hg:ArchitecturalStructure ; geo:lat ?lat ; geo:long ?long .
  OPTIONAL { ?m hg:conditionStatus ?c }
} GROUP BY ?m ?lat ?long
""")

SEVERITY = ["Good", "Restored", "Damaged", "Ruined"]          # worst last
COLOURS = {"Good": "#1a9641", "Restored": "#5e9fd1", "Damaged": "#fdae61",
           "Ruined": "#d7191c", "Not assessed": "#bbbbbb"}

def worst(conditions):
    found = [c for c in conditions.split("|") if c in SEVERITY]
    return max(found, key=SEVERITY.index) if found else "Not assessed"

points["condition"] = points["conditions"].map(worst)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.4), dpi=100, gridspec_kw={"width_ratios": [1.8, 1]},
                         layout="constrained")
valley = dict(xlim=(85.20, 85.55), ylim=(27.58, 27.80))
for ax, title in zip(axes, ["All of Nepal", "Kathmandu Valley"]):
    for cond in ["Not assessed"] + SEVERITY:
        sub = points[points["condition"] == cond]
        ax.scatter(sub["long"], sub["lat"], s=8, alpha=0.75, color=COLOURS[cond],
                   label=f"{cond} ({len(sub):,})")
    ax.set_title(title, loc="left", fontsize=12)
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_aspect(1.0)
    ax.spines[["top", "right"]].set_visible(False)
axes[1].set_xlim(*valley["xlim"]); axes[1].set_ylim(*valley["ylim"])
axes[0].legend(markerscale=2.5, fontsize=9, loc="upper right", title="Recorded condition")
fig.suptitle("Where the monuments are, and their recorded condition", fontsize=14,
             fontweight="bold", x=0.01, ha="left")
plt.show()

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**Question: how did the monuments damaged in the 2015 Gorkha earthquake fare afterwards?**

The April 2015 earthquake damaged or destroyed a large part of the Kathmandu Valley's built heritage. DANAM's monument histories describe what happened to each building, and the graph records each of those histories as an event linked to its monument. By combining those event descriptions with the condition DANAM recorded for each monument, we can see how many of the monuments that the earthquake affected have since been restored.

**Step 1: find the monuments that have an event mentioning the 2015 earthquake.** We search event descriptions for "earthquake" together with "2015".

In [ ]:
quake = sparql("""
SELECT ?m ?code ?name ?event ?description WHERE {
  ?e crm:P12_occurred_in_the_presence_of ?m ; rdfs:label ?event ; dcterms:description ?description .
  ?m a hg:ArchitecturalStructure ; rdfs:label ?name .
  OPTIONAL { ?m hg:danamCode ?code }
  FILTER(REGEX(?description, "earthquake", "i") && CONTAINS(?description, "2015"))
}
""")
print(f"{len(quake):,} events mention the 2015 earthquake, on {quake['m'].nunique():,} monuments")
quake[["code", "name", "event", "description"]].head(5)

**Step 2: look up the recorded condition of those monuments**, using the same worst-of rule as the map above, and compare them with all other monuments.

In [ ]:
condition = points.set_index("m")["condition"]
hit = condition.index.isin(quake["m"].unique())

order = SEVERITY[::-1] + ["Not assessed"]
compare = pd.DataFrame({
    "2015 earthquake in history": condition[hit].value_counts(normalize=True),
    "all other monuments": condition[~hit].value_counts(normalize=True),
}).reindex(order).fillna(0).mul(100).round(1)
print(f"{hit.sum()} earthquake-affected monuments with coordinates vs {(~hit).sum():,} others (% of each group)")
compare

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2), dpi=100)
left = pd.Series(0.0, index=compare.columns)
for cond in order:
    ax.barh(compare.columns, compare.loc[cond], left=left, color=COLOURS[cond], label=cond)
    for y, (col, v) in enumerate(compare.loc[cond].items()):
        if v >= 6:
            ax.text(left[col] + v / 2, y, f"{v:.0f}%", ha="center", va="center", fontsize=9)
    left += compare.loc[cond]
ax.set_xlim(0, 100); ax.set_xlabel("% of monuments")
ax.invert_yaxis()
ax.legend(ncol=5, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.3), frameon=False)
ax.set_title("Recorded condition: monuments hit by the 2015 earthquake vs the rest",
             loc="left", fontsize=13, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

**Answer.** In release v1.0.0, 257 monuments have an event that mentions the 2015 earthquake. **59% of them are now recorded as Restored**, twice the share among the other monuments (29%), which reflects the reconstruction effort after 2015. But about **one in five (14% Ruined, 8% Damaged) has not recovered**, and the share recorded as Ruined is higher than among the other monuments. Those are the monuments to list, map and follow up on: filter `quake` by condition to get them, with their DANAM codes and links to their DANAM pages. Change the step 1 filter from "2015" to "1934" to look at the 1934 Nepal–Bihar earthquake instead.

**Caveats.** This is a text search, so it finds monuments whose DANAM history *mentions* the earthquake; a monument that was affected but whose record does not say so is missed. Condition values are as DANAM recorded them at documentation time, not a current survey. Event types in release v1.0.0 were assigned by keyword rules, so you will see some reconstructions typed as `DestructionEvent`; the event *descriptions* used here are DANAM's own text.

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

**Challenge: build a dated timeline of Nepal's monuments.**

The events in this release **have no dates**: DANAM's event tables carry none. The dates are in the event descriptions, written in several calendar eras, often in the same sentence: Common Era (CE), Nepal Saṃvat (NS), Vikrama Saṃvat (VS) and Śaka. About 900 of the roughly 5,050 event descriptions mention one of these eras, for example *"established in 1684 CE (NS 810)"* or *"renovated again in 2007 CE (VS 2064)"*. Extracting, converting and attaching these dates would let anyone ask *when* temples were built in each part of the valley, how styles changed over the centuries, or how long monuments stood between destruction and rebuilding.

In [ ]:
# How many event descriptions mention a calendar era? A first look at the raw material.
descriptions = sparql("SELECT ?e ?d WHERE { ?e a crm:E5_Event ; dcterms:description ?d }")
era = descriptions["d"].str.extract(r"\b(CE|NS|VS|Śaka)\b", expand=False)
print(f"{era.notna().sum():,} of {len(descriptions):,} event descriptions mention an era")
print(era.value_counts().to_string())
print()
for d in descriptions.loc[era.notna(), "d"].head(3):
    print("-", d[:200])

**Advice for someone taking this on.**

- **Start with the explicit cases.** Many descriptions state a date in two eras, such as "1716 CE (NS 836)". Those pairs are ready-made test data for a converter: NS ≈ CE − 879/880, VS ≈ CE + 56/57, Śaka ≈ CE − 78.
- **Keep the uncertainty.** Dates are often approximate ("early 17th century", "around"), and a year in one era can fall across two years in another. Record a date range rather than a single year, and keep the source sentence.
- **Record how each date was obtained.** Every event already has `hg:extractionMethod "source_table"`. Give extracted dates their own provenance (method, model or rule, confidence) with PROV-O, so users can tell them apart from DANAM's own data.
- **Use what is already structured.** Inscriptions and dated objects (`hg:containsObject`), the event type and the monument's architectural style all help check whether an extracted date makes sense.
- Contributions are welcome at [CAIRNepal/HeritageGraphDataset](https://github.com/CAIRNepal/HeritageGraphDataset); its builder, ontology and SHACL shapes let you add dated events in a form that validates against the HeritageGraph ontology.

### Licence and attribution

HeritageGraph is released under the [Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0)](https://creativecommons.org/licenses/by-sa/4.0/) licence: you may share and adapt it, including commercially, if you give credit and release what you build from it under the same licence. (`metadata.json` in v1.0.0 wrongly lists CC BY 4.0; this will be corrected in the next release.)

The monument records, descriptions and images are the work of DANAM, Heidelberg University (https://danam.cats.uni-heidelberg.de); every monument links to the DANAM page it was derived from. Please credit DANAM as the source and CAIR-Nepal for the knowledge graph.